# Engenharia de Prompt

In [1]:
%%bash
pip install openrouter -q

## 1. Conexão com o modelo

In [2]:
# Libs

from google.colab import userdata
from openai import OpenAI
from IPython.display import Markdown, display

In [3]:
API_KEY = userdata.get('OPENROUTER_API_KEY')

In [4]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=API_KEY,
)

MODEL = "meta-llama/llama-3.1-8b-instruct"

## 2. Funções auxiliares

In [5]:
# Função de geração de texto

def printmd(string):
    display(Markdown(string))

def chat_generate(
    messages,
    temperature=0.2,
    top_p=0.95,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content

## 3. Configurações de Amostragem

In [6]:
prompt1 = [
    {
        "role": "user",
        "content": """Aqui estão as notas brutas de um fiscal sobre o 'Festival
        de Inverno' na Praça da Matriz: [Notas: Público estimado de 2.000
        pessoas; som ultrapassou 80 decibéis após as 22h; 5 barracas de comida
        sem alvará sanitário; presença massiva de famílias; decorações feitas
        por artesãos locais; nenhum incidente de segurança grave registrado].

        Tarefa: Com base nessas notas, escreva um texto de um parágrafo sobre
        o evento."""
    }
]

### 3.1. Cenário 1: Coerente e formal (mais estabilidade)

In [7]:
temperatura = 0.0
top_p = 0.1

for i in range(3):
  printmd(f"\n## RESPOSTA {i+1}:\n")
  printmd(chat_generate(
      prompt1,
      temperature = temperatura,
      top_p = top_p
    )
  )


## RESPOSTA 1:


O Festival de Inverno na Praça da Matriz foi um sucesso, com um público estimado de 2.000 pessoas. Embora o som tenha ultrapassado os 80 decibéis após as 22h, o evento foi marcado por uma atmosfera agradável, com decorações feitas por artesãos locais e a presença massiva de famílias. Além disso, o evento não registrou nenhum incidente de segurança grave. No entanto, houve cinco barracas de comida que não tinham alvará sanitário, o que poderia ter causado problemas de saúde pública.


## RESPOSTA 2:


O Festival de Inverno na Praça da Matriz foi um sucesso de público, com cerca de 2.000 pessoas presentes. A programação musical foi animada, mas ultrapassou os limites de decibéis após as 22h. Além disso, foram identificadas 5 barracas de comida sem alvará sanitário, o que pode ter causado preocupações de saúde pública. No entanto, a presença de famílias foi massiva, o que é um sinal de que o evento foi bem recebido pela comunidade. As decorações, feitas por artesãos locais, foram um destaque do evento. Felizmente, não houve incidentes de segurança graves registrados.


## RESPOSTA 3:


O Festival de Inverno na Praça da Matriz foi um sucesso de público, com cerca de 2.000 pessoas presentes. A programação musical foi animada, mas ultrapassou os limites de decibéis após as 22h. Além disso, foram identificadas 5 barracas de comida sem alvará sanitário, o que pode ter gerado preocupações de saúde pública. No entanto, a presença de famílias foi massiva, o que é um sinal de que o evento foi bem recebido pela comunidade. As decorações, feitas por artesãos locais, foram um destaque do evento. Felizmente, não houve incidentes de segurança graves registrados.

### 3.2. Cenário 2: Criativo (mais diversidade)

In [8]:
temperatura = 1.0
top_p = 1.0

for i in range(3):
  printmd(f"\n## RESPOSTA {i+1}:\n")
  printmd(chat_generate(
      prompt1,
      temperature = temperatura,
      top_p = top_p
    )
  )


## RESPOSTA 1:


O Festival de Inverno na Praça da Matriz foi um desses eventos festivos
que geraram forte sensação de abrandamento nas redes sociais. Com um público desvio de 2.000 pessoas, atordoou o bairro com decibéis acima de oitenta após as vinte e duas da noite. Durante o dia, 5 (cinco) barracas albergaram
articulações gastronômicas sem Carta Sanitária e dentro da festa puderam-se observar as cores pinceladas de artesanos da praça, focando na organização de sua decoração. Reforçando a alegria da tarde o ocorreram até incidente mortífero revigorando a energia festiva.


## RESPOSTA 2:


O Festival de Inverno na Praça da Matriz foi um sucesso com um público estimado de 2000 pessoas, conjugando diversão e diversidade com decorações criadas por artesãos locais, além de uma presença massiva de famílias. O evento contou com a participação de 5 barracas de comida, mas só cinco delas estavam legalizadas, instaurando problemas de controle sanitário dentre otherwise. Além disso, a onda oxidante da música ultrapassou os 80 decibeis após o horário: 10 e cruzou sozinha as 22h, mas nenhum incidente de segurança grave foi registrado.


## RESPOSTA 3:


O Festival de Inverno teve uma boa aceitação, alcançando um público multiplicado que gira em torno de 2.000 pessoas; porém, existe um cheque especial a ser depositado relativa ao ruído excessivo. Foi identificada a presença de cinco barracas temporárias implantadas sob a ausência do alvará de vigilância sanitária.

## 4. Arquitetura de Prompt

### 4.1. Cenário 1 (Fiscalização)

In [ ]:
# System Prompt (Propósito e Formato)
system_prompt2 = '''
PROPÓSITO E FORMATO: Você é um sistema especializado em processamento de dados
administrativos e conformidade regulatória. Sua função é transformar anotações
de campo desestruturadas em relatórios técnicos precisos, garantindo que nenhum
dado factual seja omitido ou alterado. *Sua saída deve ser estritamente em
formato JSON válido*. Não inclua informações fora do JSON.

*Esquema JSON obrigatório:*

{
  "evento": string,
  "publico_alvo": string,
  "volume_estimado": int,
  "infracoes_detectadas": [
    {
      "tipo": string,
      "descricao": string,
      "risco_regulatorio": string
    }
  ],
  "balanco_seguranca": string
}
'''

# Role Prompt (Persona e Tom):
role_prompt2 = '''
PERSONA: Aja como um *Auditor de Serviços Públicos sênior*. Adote um tom
formal, imparcial e analítico. Sua expertise deve focar em identificar riscos
sanitários, conformidade sonora e segurança pública, priorizando a clareza
quanto ao impacto regulatório.
'''

# Contextual Prompt (Dados Variáveis)
context_prompt2 = '''
CONTEXTO: Fiscalização realizada no 'Festival de Inverno' na Praça da Matriz.

NOTAS BRUTAS: {Público estimado de 2.000 pessoas; som ultrapassou 80 decibéis
após as 22h; 5 barracas de comida sem alvará sanitário; presença massiva de
famílias; decorações feitas por artesãos locais; nenhum incidente de segurança
grave registrado}.
'''

# Tarefa e instruções
task_instructions2 = '''
TAREFA: *Analise* as notas acima e *gere* um resumo técnico estruturado em JSON.

*INSTRUÇÕES:* Liste as métricas de público, categorize as infrações por tipo
(sanitária/sonora) e descreva o impacto social positivo observado.
'''

In [ ]:
prompt2 = [
    {
        "role": "system",
        "content": f"""
{system_prompt2}

{role_prompt2}

"""
    },
    {
        "role": "user",
        "content": f"""
{context_prompt2}

{task_instructions2}

Agora execute a tarefa.
"""
    }
]

In [ ]:
temperatura = 0.1
top_p = 0.9

for i in range(2):
  printmd(f"\n## RESPOSTA {i+1}:\n")
  printmd(chat_generate(
      prompt2,
      temperature = temperatura,
      top_p = top_p
    )
  )


## RESPOSTA 1:


**Resumo Técnico do Fiscalização do Festival de Inverno**

```json
{
  "evento": "Festival de Inverno",
  "publico_alvo": "Praça da Matriz",
  "volume_estimado": 2000,
  "infracoes_detectadas": [
    {
      "tipo": "Sanitária",
      "descricao": "5 barracas de comida sem alvará sanitário",
      "risco_regulatorio": "Risco de contaminação alimentar"
    },
    {
      "tipo": "Sonora",
      "descricao": "Som ultrapassou 80 decibéis após as 22h",
      "risco_regulatorio": "Risco de danos auditivos"
    }
  ],
  "balanco_seguranca": "Nenhum incidente de segurança grave registrado"
}
```

**Análise e Observações**

*   O público estimado foi de 2.000 pessoas, o que indica uma grande concentração de pessoas na área.
*   As infrações detectadas foram categorizadas em sanitária e sonora. A falta de alvará sanitário em 5 barracas de comida representa um risco significativo de contaminação alimentar. Além disso, o som ultrapassando 80 decibéis após as 22h representa um risco de danos auditivos.
*   A presença massiva de famílias na área é um indicativo de que o evento foi bem recebido pela comunidade local.
*   A decoração feita por artesãos locais é um exemplo de apoio ao desenvolvimento local e à economia regional.
*   O fato de nenhum incidente de segurança grave ter sido registrado é um indicativo de que as medidas de segurança implementadas foram eficazes.


## RESPOSTA 2:


**Resumo Técnico do Fiscalização do Festival de Inverno**

```json
{
  "evento": "Festival de Inverno",
  "publico_alvo": "Praça da Matriz",
  "volume_estimado": 2000,
  "infracoes_detectadas": [
    {
      "tipo": "sanitária",
      "descricao": "5 barracas de comida sem alvará sanitário",
      "risco_regulatorio": "Risco de contaminação alimentar"
    },
    {
      "tipo": "sonora",
      "descricao": "Som ultrapassou 80 decibéis após as 22h",
      "risco_regulatorio": "Risco de danos auditivos"
    }
  ],
  "balanco_seguranca": "Nenhum incidente de segurança grave registrado"
}
```

**Análise e Observações**

*   O público estimado foi de 2.000 pessoas, o que indica uma grande concentração de pessoas na área.
*   As infrações detectadas foram categorizadas em sanitária e sonora.
*   A presença de 5 barracas de comida sem alvará sanitário representa um risco de contaminação alimentar, o que pode ter consequências graves para a saúde pública.
*   O som ultrapassando 80 decibéis após as 22h representa um risco de danos auditivos, especialmente para as pessoas que estavam próximas ao local.
*   A presença massiva de famílias no evento é um indicativo de que o festival foi bem recebido pela comunidade local.
*   A decoração feita por artesãos locais é um exemplo de apoio ao desenvolvimento local e à economia comunitária.
*   A ausência de incidentes de segurança graves é um indicativo de que as medidas de segurança implementadas foram eficazes.

### 4.2. Cenário 2 (Marketing Institucional)

In [ ]:
# System Prompt (Propósito e Formato)
system_prompt3 = '''
PROPÓSITO E FORMATO: Você é um especialista em *Comunicação Social e Marketing
Governamental*. Sua função é transformar dados brutos de fiscalização em
eventos em *narrativas institucionais inspiradoras* que valorizem a imagem da
prefeitura e o bem-estar da comunidade. Você deve garantir que a saída seja
rica em detalhes, mas extremamente concisa. *Sua saída deve ser estritamente em
formato JSON válido* para  ser consumida por plataformas de redes sociais e
sites oficiais. Não inclua explicações fora do JSON.

*Esquema JSON obrigatório:*
{
  "manchete_impacto": string,
  "corpo_texto_denso": string,
  "destaque_comunitario": string,
  "hashtags_sugeridas": string
}
'''

# Role Prompt (Persona e Tom):
role_prompt3 = '''
PERSONA: Aja como um *Redator Publicitário Criativo*. Adote um tom *Inspirador,
Persuasivo e Influente*. Sua expertise deve focar em destacar o sucesso do
evento, a segurança das famílias e o apoio à economia local, transformando
números em conquistas sociais.
'''

# Contextual Prompt (Dados Variáveis)
context_prompt3 = '''
CONTEXTO: Fiscalização realizada no 'Festival de Inverno' na Praça da Matriz.

NOTAS BRUTAS: {Público estimado de 2.000 pessoas; som ultrapassou 80 decibéis
após as 22h; 5 barracas de comida sem alvará sanitário; presença massiva de
famílias; decorações feitas por artesãos locais; nenhum incidente de segurança
grave registrado}.
'''

# Tarefa e instruções
task_instructions3 = '''
TAREFA: Crie uma síntese narrativa do evento.

INSTRUÇÕES: Foque na harmonia e no sucesso do evento; Use verbos de ação como
'Celebrar', 'Fortalecer' e 'Encantar'.
'''

In [ ]:
prompt3 = [
    {
        "role": "system",
        "content": f"""
{system_prompt3}

{role_prompt3}

"""
    },
    {
        "role": "user",
        "content": f"""
{context_prompt3}

{task_instructions3}

Agora execute a tarefa.
"""
    }
]

In [ ]:
temperatura = 0.9
top_p = 0.99

for i in range(2):
  printmd(f"\n## RESPOSTA {i+1}:\n")
  printmd(chat_generate(
      prompt3,
      temperature = temperatura,
      top_p = top_p
    )
  )


## RESPOSTA 1:


{
  "manchete_impacto": "Festival de Inverno: Celebração Harmoniosa em Plena Praça da Matriz",
  "corpo_texto_denso": "Nossa comunidade reuniu-se na Praça da Matriz para celebrar o inverno em um ambiente de alegria e confraternização. Com um público estimado de 2.000 pessoas, o evento foi um verdadeiro sucesso, encantando todos que participaram. Ao longo da noite, a decoração feita por artesãos locais iluminou o espaço, enquanto a presença massiva de famílias reforçou a importância da integração comunitária. Além disso, a harmoniosa qualidade do som, gerenciado dentro dos parâmetros legais, garantindo que a vibração fosse única sem comprometer a saúde auditiva dos nossos habitantes. Um total de 5 barracas de comida adicionaram uma variação deliciosa às opções da noite.",
  "destaque_comunitario": "O Festival de Inverno foi mais do que um simples evento, foi uma oportunidade de fortalecer os laços entre nossos habitantes e entre a nossa comunidade e o comércio local, encorajando a economia local e oferecendo momentos inesquecíveis para todas as famílias.",
  "hashtags_sugeridas": "#FestivalDeInverno #CelebraçãoHarmoniosa #IntegraçãoComunitária #EconomiaLocal #FamíliaEmAção"
}


## RESPOSTA 2:


{
  "manchete_impacto": "Festival de Inverno Celebra Harmonia e Féria Local",
  "corpo_texto_denso": "Mais de 2.000 pessoas se reuniram na Praça da Matriz para celebrar o Festival de Inverno! O som vibrante da música encantou a multidão, mas os nossos agentes de fiscalização estiveram presentes, garantindo que a decibéis não ultrapassassem o limite. Assim, a massa de famílias presente teve um tempo incrível para curtir a festa. Além disso, barracas de comida locais fortaleceram a economia da região e, visando seu sucesso, optaram em comprar alvarás para garantir a sanção da saúde geral da população, em outras palavras "alvarás sanitários". E como se isso não fosse o suficiente, zero incidentes de segurança foram registrados durante a festa, garantindo a todos presentes um Domingo Invernal seguro e inesquecível.",
  "destaque_comunitario": "Criatividade e engenhosidade de nossos habitantes locais, tornando essa festa uma característica icônica da praça da Matriz.",
  "hashtags_sugeridas": "#FestivalDeInverno #PraçaDaMatriz #ComunidadeFortalecida #FestasSeguras #AlvarásSanitários #EconomiaLocal #InvernoEncantador"
}

## 5. Zero-shot, One-shot e Few-shot Prompting

Incremente o prompt com persona "Auditor de serviços públicos sênior" (zero-shot), com prompts one-shot e few-shot.

### 5.1. One-shot

In [ ]:
# System Prompt (Propósito e Formato)
system_prompt4 = '''
PROPÓSITO E FORMATO: Você é um sistema especializado em processamento de dados
administrativos e conformidade regulatória. Sua função é transformar anotações
de campo desestruturadas em relatórios técnicos precisos, garantindo que nenhum
dado factual seja omitido ou alterado. *Sua saída deve ser estritamente em
formato JSON válido*. Não inclua informações fora do JSON.

REGRAS OBRIGATÓRIAS:
1. Responda exclusivamente com JSON válido.
2. Não inclua texto antes ou depois do JSON.
3. Não use markdown.
4. Não inclua comentários.
5. A resposta deve começar com { e terminar com }.
6. Não adicione chaves extras.
7. Não omita chaves.

*Esquema JSON obrigatório:*

{
  "evento": string,
  "publico_alvo": string,
  "volume_estimado": int,
  "infracoes_detectadas": [
    {
      "tipo": string,
      "descricao": string,
      "risco_regulatorio": string
    }
  ],
  "balanco_seguranca": string
}
'''

exemplo1 = """
*EXEMPLO:*

CONTEXTO: Fiscalização realizada na "Feira Noturna Municipal".

NOTAS BRUTAS: {Público estimado de 800 pessoas; música ambiente dentro do limite
legal de 60 decibéis; 2 barracas sem licença sanitária; presença predominante
de adultos; nenhuma ocorrência policial registrada}.

Saída esperada para o exemplo:

{
  "evento": "Feira Noturna Municipal",
  "publico_alvo": "Predominância de adultos",
  "volume_estimado": 800,
  "infracoes_detectadas": [
    {
      "tipo": "Sanitária",
      "descricao": "Duas barracas operando sem licença sanitária",
      "risco_regulatorio": "Potencial risco à saúde pública e descumprimento de
      normas municipais"
    }
  ],
  "balanco_seguranca": "Não houve ocorrências policiais ou incidentes graves."
}

--- FIM DO EXEMPLO ---
"""

In [ ]:
prompt4 = [
    {
        "role": "system",
        "content": f"""
{system_prompt4}

{role_prompt2}

"""
    },
    {
        "role": "user",
        "content": f"""
{exemplo1}

Agora execute a tarefa real abaixo:

{context_prompt2}

{task_instructions2}
"""
    }
]

In [ ]:
temperatura = 0.2
top_p = 0.9

printmd(chat_generate(
    prompt4,
    temperature = temperatura,
    top_p = top_p
  )
)

{
  "evento": "Festival de Inverno",
  "publico_alvo": "Famílias",
  "volume_estimado": 2000,
  "infracoes_detectadas": [
    {
      "tipo": "Sanitária",
      "descricao": "Cinco barracas de comida operando sem alvará sanitário",
      "risco_regulatorio": "Potencial risco à saúde pública e descumprimento de normas municipais"
    },
    {
      "tipo": "Sonora",
      "descricao": "Som ultrapassou 80 decibéis após as 22h",
      "risco_regulatorio": "Potencial risco à saúde auditiva e descumprimento de normas municipais"
    }
  ],
  "balanco_seguranca": "Não houve incidentes de segurança graves registrados e a presença de decorações feitas por artesãos locais foi um impacto social positivo observado."
}

### 5.2. Few-shot

In [ ]:
exemplo2 = '''
CONTEXTO: Fiscalização na "Feira Gastronômica Municipal".

NOTAS BRUTAS: {Público estimado de 1.200 pessoas; som ambiente dentro do limite
legal; 3 barracas sem licença sanitária; público majoritariamente adulto;
nenhuma ocorrência policial}.

SAÍDA:

{
  "evento": "Feira Gastronômica Municipal",
  "publico_alvo": "Predominância de adultos",
  "volume_estimado": 1200,
  "infracoes_detectadas": [
    {
      "tipo": "Sanitária",
      "descricao": "Três barracas operando sem licença sanitária",
      "risco_regulatorio": "Descumprimento das normas sanitárias municipais com
      potencial risco à saúde pública"
    }
  ],
  "balanco_seguranca": "Evento transcorreu sem ocorrências policiais ou
  incidentes graves."
}
'''

exemplo3 = '''
CONTEXTO: Fiscalização no "Show Cultural da Juventude".

NOTAS BRUTAS: {Público estimado de 3.500 pessoas; som atingiu 95 decibéis após
as 23h; todas as barracas com alvará regular; presença significativa de jovens;
registro de 1 atendimento médico leve}.

SAÍDA:

{
  "evento": "Show Cultural da Juventude",
  "publico_alvo": "Predominância de jovens",
  "volume_estimado": "Aproximadamente 3.500 participantes",
  "infracoes_detectadas": [
    {
      "tipo": "Sonora",
      "descricao": "Emissão sonora acima de 95 decibéis após as 23h",
      "risco_regulatorio": "Descumprimento da legislação municipal de controle
      de ruído"
    }
  ],
  "balanco_seguranca": "Evento com um atendimento médico leve e sem incidentes
  de segurança grave."
}
'''

In [ ]:
prompt5 = [
    {
        "role": "system",
        "content": f"""
{system_prompt4}

{role_prompt2}

"""
    },
    {
        "role": "user",
        "content": f"""
EXEMPLO 1:
{exemplo2}

EXEMPLO 2:
{exemplo3}

Agora execute a tarefa real:

{context_prompt2}

{task_instructions2}
"""
    }
]

In [ ]:
temperatura = 0.2
top_p = 0.9

printmd(chat_generate(
    prompt5,
    temperature = temperatura,
    top_p = top_p
  )
)

{
  "evento": "Festival de Inverno",
  "publico_alvo": "Famílias",
  "volume_estimado": 2000,
  "infracoes_detectadas": [
    {
      "tipo": "Sanitária",
      "descricao": "Cinco barracas de comida operando sem alvará sanitário",
      "risco_regulatorio": "Descumprimento das normas sanitárias municipais com
      potencial risco à saúde pública"
    },
    {
      "tipo": "Sonora",
      "descricao": "Emissão sonora acima de 80 decibéis após as 22h",
      "risco_regulatorio": "Descumprimento da legislação municipal de controle
      de ruído"
    }
  ],
  "balanco_seguranca": "Evento com decorações feitas por artesãos locais e sem
  incidentes de segurança grave, destacando a presença de famílias e a
  contribuição para a economia local."
}

## 6. Chain-of-Thought (CoT)

### 6.1. Prompt simples

In [ ]:
prompt6 = messages = [
  {
    "role": "system",
    "content": """Você é um assistente técnico de auditoria fiscal tributária.
    Forneça respostas objetivas, numéricas e diretamente aplicáveis. Não
    explique o raciocínio. Apresente apenas o valor final solicitado."""
  },
  {
    "role": "user",
    "content": """Calcule o valor total devido (imposto + multa) para a Empresa
    Y com base nos dados abaixo:

Receita Bruta: R$ 1.450.000,00
Alíquota Progressiva:
- 2% sobre os primeiros R$ 500.000,00
- 5% sobre o valor excedente

Dedução permitida: R$ 120.000,00 (aplicada sobre o total do imposto calculado).
Multa: 1% ao mês (juros simples) sobre o valor líquido devido.
Período de atraso: 5 meses.

Retorne apenas:
Valor total a recolher: R$ X.XXX.XXX,XX
"""
  }
]

In [ ]:
temperatura = 0.0
top_p = 1.0         # Irrelevante!

resposta = chat_generate(
    prompt6,
    temperature = temperatura,
    top_p = top_p
)

In [ ]:
print(resposta)

**Calculo do Imposto:**

- Valor sobre os primeiros R$ 500.000,00: R$ 500.000,00 * 2% = R$ 10.000,00
- Valor excedente: R$ 1.450.000,00 - R$ 500.000,00 = R$ 950.000,00
- Imposto sobre o valor excedente: R$ 950.000,00 * 5% = R$ 47.500,00
- Total do imposto: R$ 10.000,00 + R$ 47.500,00 = R$ 57.500,00

**Calculo da Dedução:**

- Dedução permitida: R$ 120.000,00
- Imposto líquido devido: R$ 57.500,00 - R$ 120.000,00 = R$ -62.500,00 (não é possível, pois não pode haver débito)
- Imposto líquido devido: R$ 57.500,00 (não é possível aplicar a dedução)

**Calculo da Multa:**

- Valor líquido devido: R$ 57.500,00
- Juros simples ao mês: R$ 57.500,00 * 1% = R$ 575,00
- Período de atraso: 5 meses
- Multa total: R$ 575,00 * 5 = R$ 2.875,00

**Calculo do Valor Total a Recolher:**

- Imposto líquido devido: R$ 57.500,00
- Multa total: R$ 2.875,00
- Valor total a recolher: R$ 57.500,00 + R$ 2.875,00 = R$ 60.375,00


### 6.2. Prompt com CoT

In [ ]:
prompt7 = [
  {
    "role": "system",
    "content": """Você é um Auditor Fiscal Sênior especialista em cálculos
    tributários complexos.
Sua prioridade é precisão matemática absoluta.
Antes de apresentar o valor final, você deve estruturar explicitamente todas as
etapas de cálculo.
Apresente cada etapa numerada e com valores intermediários claros."""
  },
  {
    "role": "user",
    "content": """Realize a auditoria fiscal da Empresa Y.

DADOS:
- Receita Bruta: R$ 1.450.000,00
- Alíquota Progressiva:
  • 2% até R$ 500.000,00
  • 5% sobre o excedente
- Dedução: R$ 120.000,00 (aplicada após cálculo total do imposto)
- Juros: 1% ao mês (juros simples)
- Atraso: 5 meses

INSTRUÇÕES OBRIGATÓRIAS:

1. Separe explicitamente a receita nas duas faixas de tributação.
2. Calcule o imposto de cada faixa individualmente.
3. Some os impostos para obter o imposto bruto total.
4. Subtraia a dedução e identifique o valor líquido devido.
5. Calcule os juros simples acumulados (1% x 5 meses).
6. Some o valor líquido e os juros.
7. Revise o resultado final antes de apresentá-lo.

Apresente:
- Cada etapa numerada
- Valores intermediários
- Fórmulas utilizadas
- Resultado final destacado
"""
  }
]

In [ ]:
temperatura = 0.0
top_p = 1.0         # Irrelevante!

resposta = chat_generate(
    prompt7,
    temperature = temperatura,
    top_p = top_p,
    max_tokens=1000
)

In [ ]:
print(resposta)

**Auditoria Fiscal da Empresa Y**

**DADOS**

- Receita Bruta: R$ 1.450.000,00
- Alíquota Progressiva:
  • 2% até R$ 500.000,00
  • 5% sobre o excedente
- Dedução: R$ 120.000,00 (aplicada após cálculo total do imposto)
- Juros: 1% ao mês (juros simples)
- Atraso: 5 meses

**ETAPAS DE CÁLCULO**

1. **Separe a receita nas duas faixas de tributação**

Receita até R$ 500.000,00: R$ 500.000,00
Receita excedente: R$ 1.450.000,00 - R$ 500.000,00 = R$ 950.000,00

2. **Calcule o imposto de cada faixa individualmente**

Imposto até R$ 500.000,00: 2% de R$ 500.000,00 = 0,02 x R$ 500.000,00 = R$ 10.000,00
Imposto sobre o excedente: 5% de R$ 950.000,00 = 0,05 x R$ 950.000,00 = R$ 47.500,00

3. **Some os impostos para obter o imposto bruto total**

Imposto bruto total: R$ 10.000,00 + R$ 47.500,00 = R$ 57.500,00

4. **Subtraia a dedução e identifique o valor líquido devido**

Valor líquido devido: R$ 57.500,00 - R$ 120.000,00 = R$ -62.500,00 (não é possível ter um valor negativo, portanto, o valor lí

## 7. Chain-of-Verification (CoV)

In [ ]:
prompt8 = [
    {
        "role": "user",
        "content": """Cite alguns políticos que nasceram no estado de Pernambuco"""
    }
]

In [ ]:
temperatura = 0.2
top_p = 0.9

resposta = chat_generate(
    prompt8,
    temperature = temperatura,
    top_p = top_p
)

In [ ]:
printmd(resposta)

Aqui estão alguns políticos que nasceram no estado de Pernambuco:

1. **Joaquim Nabuco** (1849-1910): Nasceu em Recife, foi um político, escritor e abolicionista brasileiro.
2. **Manuel Borba** (1858-1928): Nasceu em Recife, foi um político e advogado brasileiro, que serviu como governador de Pernambuco.
3. **José Mariano da Silva** (1862-1924): Nasceu em Recife, foi um político e advogado brasileiro, que serviu como governador de Pernambuco.
4. **Manuel de Albuquerque** (1873-1943): Nasceu em Recife, foi um político e advogado brasileiro, que serviu como governador de Pernambuco.
5. **Pedro Gondim** (1895-1965): Nasceu em Recife, foi um político e advogado brasileiro, que serviu como governador de Pernambuco.
6. **João Cleofas** (1901-1976): Nasceu em Recife, foi um político e advogado brasileiro, que serviu como governador de Pernambuco.
7. **Mário Negromonte** (1954-): Nasceu em Recife, é um político e engenheiro brasileiro, que serviu como deputado federal e governador de Sergipe.
8. **João Paulo** (1956-): Nasceu em Recife, é um político e advogado brasileiro, que serviu como deputado federal e governador de Pernambuco.
9. **Marília Arraes** (1961-): Nasceu em Recife, é uma política e advogada brasileira, que serviu como deputada federal e vereadora de Recife.
10. **Raquel Lyra** (1973-): Nasceu em Recife, é uma política e advogada brasileira, que serviu como deputada federal e vereadora de Recife.

Essa é apenas uma lista parcial de políticos que nasceram em Pernambuco. Existem muitos outros que também contribuíram para a política do estado e do país.

In [ ]:
prompt9 = [
    {
        "role": "system",
        "content": """Você é um assistente especializado em verificação factual e precisão histórica.
Sempre que listar fatos, deve justificar e verificar cada afirmação antes de apresentar o resultado final."""
    },
    {
        "role": "user",
        "content": """
Cite alguns políticos que nasceram no estado de Pernambuco.

INSTRUÇÕES OBRIGATÓRIAS:

1. Primeiro, elabore uma lista preliminar de políticos brasileiros que você acredita terem nascido em Pernambuco.
2. Para cada nome <NOME> da lista, responda a seguinte pergunta: Em que estado nasceu <NOME>?
3. Em seguida, revise cada item da lista preliminar gerada no passo 1 de acordo com as respostas do passo 2
4. Gere uma lista final revisada contendo apenas os nomes confirmados.

Organize sua resposta nas seguintes seções:

  - SEÇÃO 1: Lista Inicial
  - SEÇÃO 2: Verificação Item a Item
  - SEÇÃO 3: Lista Final Confirmada
"""
    }
]

In [ ]:
temperatura = 0.2
top_p = 0.9

resposta = chat_generate(
    prompt9,
    temperature = temperatura,
    top_p = top_p
)

In [ ]:
printmd(resposta)

**SEÇÃO 1: Lista Inicial**

Aqui está uma lista de políticos brasileiros que eu acredito terem nascido em Pernambuco:

1. Fernando Collor de Mello
2. Miguel Arraes
3. Miguel Calmon
4. João Lyra Filho
5. Fernando Bezerra Coelho

**SEÇÃO 2: Verificação Item a Item**

Vou verificar cada item da lista inicial:

1. **Fernando Collor de Mello**: De acordo com a biografia oficial, Fernando Collor de Mello nasceu em Rio de Janeiro, Rio de Janeiro.
2. **Miguel Arraes**: Miguel Arraes nasceu em Recife, Pernambuco.
3. **Miguel Calmon**: Miguel Calmon nasceu em Recife, Pernambuco.
4. **João Lyra Filho**: João Lyra Filho nasceu em Recife, Pernambuco.
5. **Fernando Bezerra Coelho**: Fernando Bezerra Coelho nasceu em Recife, Pernambuco.

**SEÇÃO 3: Lista Final Confirmada**

Aqui está a lista final de políticos que nasceram em Pernambuco:

1. Miguel Arraes
2. Miguel Calmon
3. João Lyra Filho
4. Fernando Bezerra Coelho

Esses políticos confirmados nasceram em Recife, Pernambuco.

## 8. Chain-of-Density (CoD)

### 8.1. Sem CoD

In [ ]:
prompt10 = [
    {
        "role": "system",
        "content": """
Você é um assistente especializado em sumarização jurídica de alta precisão.
"""
    },
    {
        "role": "user",
        "content": """
<<<
Nos termos do artigo 5º, inciso XXXV, da Constituição Federal, estabelece-se como garantia fundamental que a lei não excluirá da apreciação do Poder Judiciário lesão ou ameaça a direito. Trata-se de cláusula pétrea que assegura a todos os cidadãos o acesso à jurisdição sempre que houver controvérsia acerca de direitos individuais ou coletivos. Essa norma consagra o princípio da inafastabilidade da jurisdição, impedindo que atos administrativos ou disposições legais obstem o controle judicial de eventuais ilegalidades ou abusos cometidos pelo Estado ou por particulares no exercício de funções públicas.

No caso concreto, o autor formulou requerimento administrativo perante a Administração Pública, buscando o reconhecimento de determinado direito que entende lhe assistir. Contudo, a autoridade administrativa competente negou provimento ao pedido, fundamentando sua decisão na ocorrência de prescrição. Segundo o entendimento adotado pela Administração, teria transcorrido o prazo legal para o exercício da pretensão do interessado, o que inviabilizaria a análise do mérito do pleito. A decisão administrativa limitou-se a reconhecer o suposto decurso do tempo como causa impeditiva do exame substancial do direito alegado.

O autor, por sua vez, sustenta que não foi regularmente notificado dos atos administrativos essenciais ao curso do procedimento, especialmente daqueles que poderiam dar início à contagem do prazo prescricional. Argumenta que a ausência de notificação válida comprometeu sua ciência inequívoca dos fatos e dos fundamentos utilizados pela Administração, configurando cerceamento de defesa. Segundo sua tese, sem a devida comunicação formal e pessoal, não se poderia considerar iniciado o prazo para eventual impugnação ou adoção de medidas judiciais cabíveis, razão pela qual a alegada prescrição não poderia prevalecer.

Ao submeter a controvérsia ao Poder Judiciário, o juízo de primeiro grau entendeu que não se tratava propriamente de prescrição, mas de decadência do direito. Fundamentou sua decisão no prazo quinquenal previsto no artigo 54 do Código de Defesa do Consumidor, aplicando-o ao caso concreto por analogia ou por entender configurada relação jurídica compatível com a norma invocada. Com base nessa interpretação, concluiu que o direito do autor estaria fulminado pelo decurso do prazo de cinco anos, extinguindo o processo com resolução de mérito, nos termos da legislação processual vigente.

O autor, inconformado com a sentença, afirma que a decisão judicial incorreu em violação a princípios constitucionais basilares, notadamente o devido processo legal, a ampla defesa e o contraditório. Sustenta que a extinção do processo sem a adequada apreciação das alegações relativas à ausência de notificação e à controvérsia sobre o termo inicial do prazo implicou restrição indevida ao seu direito de acesso à justiça. Alega, ainda, que a aplicação automática do prazo decadencial, sem exame aprofundado das circunstâncias fáticas e jurídicas, afronta a garantia de tutela jurisdicional efetiva assegurada pela Constituição.
>>>

Resuma o texto jurídico acima em até 150 palavras.
"""
    }
]

In [ ]:
temperatura = 0.1
top_p = 0.9

resposta = chat_generate(
    prompt10,
    temperature = temperatura,
    top_p = top_p
)

In [ ]:
printmd(resposta)

O autor formulou requerimento administrativo para reconhecimento de direito, mas a Administração negou, fundamentando em prescrição. O autor alega que não foi notificado regularmente, o que comprometeu sua defesa. O juízo de primeiro grau entendeu que não se tratava de prescrição, mas de decadência do direito, aplicando o prazo quinquenal do Código de Defesa do Consumidor. A decisão extinguiu o processo com resolução de mérito.

O autor alega violação a princípios constitucionais, incluindo o devido processo legal, ampla defesa e contraditório. Sustenta que a extinção do processo sem apreciação das alegações sobre notificação e termo inicial do prazo implicou restrição ao direito de acesso à justiça. Alega que a aplicação automática do prazo decadencial afronta a garantia de tutela jurisdicional efetiva.

### 8.2. Com CoD

In [ ]:
prompt11 = [
    {
        "role": "system",
        "content": """
Você é um assistente especializado em sumarização jurídica de alta precisão.
Aplique a técnica Chain-of-Density (CoD), organizando o conteúdo por unidades temáticas,
classificando densidade informacional e produzindo sínteses progressivamente mais densas.
Seja rigoroso na estrutura e evite redundâncias.

OBJETIVO:
Produzir um resumo jurídico de alta densidade utilizando a técnica Chain-of-Density (CoD),
com refinamento iterativo.

PASSOS OBRIGATÓRIOS:

1. SEGMENTAÇÃO TEMÁTICA
   - Divida o texto em unidades tópicas.
   - Dê um título a cada unidade.
   - Resuma cada uma brevemente.

2. CLASSIFICAÇÃO DE DENSIDADE
   - Classifique cada unidade como:
     Alta (núcleo argumentativo),
     Média (contexto relevante),
     Baixa (detalhe subsidiário).
   - Justifique brevemente a classificação.

3. AGREGAÇÃO (CoD - Iteração 1)
   - Produza um resumo consolidado priorizando unidades de Alta e Média densidade.
   - Limite: até 300 palavras.

4. ITERAÇÃO PROGRESSIVA (Repetir 5x)
   - A partir do resumo anterior, produza nova versão:
     * mais concisa
     * mais informacionalmente densa
     * sem perder elementos jurídicos essenciais
   - Cada nova versão deve reduzir redundâncias e aumentar compressão sem sacrificar precisão.
   - Resumo final deve conter no máximo 150 palavras.

5. Repita o processo de refinamento até completar 5 versões.

FORMATO DE SAÍDA:

SEÇÃO A - Segmentação Temática
SEÇÃO B - Classificação de Densidade

SEÇÃO C - Iterações de Resumo:
Resumo 1:
Resumo 2:
Resumo 3:
Resumo 4:
Resumo 5:

Cada resumo deve ser exibido integralmente.
"""
    },
    {
        "role": "user",
        "content": """
<<<
Nos termos do artigo 5º, inciso XXXV, da Constituição Federal, estabelece-se como garantia fundamental que a lei não excluirá da apreciação do Poder Judiciário lesão ou ameaça a direito. Trata-se de cláusula pétrea que assegura a todos os cidadãos o acesso à jurisdição sempre que houver controvérsia acerca de direitos individuais ou coletivos. Essa norma consagra o princípio da inafastabilidade da jurisdição, impedindo que atos administrativos ou disposições legais obstem o controle judicial de eventuais ilegalidades ou abusos cometidos pelo Estado ou por particulares no exercício de funções públicas.

No caso concreto, o autor formulou requerimento administrativo perante a Administração Pública, buscando o reconhecimento de determinado direito que entende lhe assistir. Contudo, a autoridade administrativa competente negou provimento ao pedido, fundamentando sua decisão na ocorrência de prescrição. Segundo o entendimento adotado pela Administração, teria transcorrido o prazo legal para o exercício da pretensão do interessado, o que inviabilizaria a análise do mérito do pleito. A decisão administrativa limitou-se a reconhecer o suposto decurso do tempo como causa impeditiva do exame substancial do direito alegado.

O autor, por sua vez, sustenta que não foi regularmente notificado dos atos administrativos essenciais ao curso do procedimento, especialmente daqueles que poderiam dar início à contagem do prazo prescricional. Argumenta que a ausência de notificação válida comprometeu sua ciência inequívoca dos fatos e dos fundamentos utilizados pela Administração, configurando cerceamento de defesa. Segundo sua tese, sem a devida comunicação formal e pessoal, não se poderia considerar iniciado o prazo para eventual impugnação ou adoção de medidas judiciais cabíveis, razão pela qual a alegada prescrição não poderia prevalecer.

Ao submeter a controvérsia ao Poder Judiciário, o juízo de primeiro grau entendeu que não se tratava propriamente de prescrição, mas de decadência do direito. Fundamentou sua decisão no prazo quinquenal previsto no artigo 54 do Código de Defesa do Consumidor, aplicando-o ao caso concreto por analogia ou por entender configurada relação jurídica compatível com a norma invocada. Com base nessa interpretação, concluiu que o direito do autor estaria fulminado pelo decurso do prazo de cinco anos, extinguindo o processo com resolução de mérito, nos termos da legislação processual vigente.

O autor, inconformado com a sentença, afirma que a decisão judicial incorreu em violação a princípios constitucionais basilares, notadamente o devido processo legal, a ampla defesa e o contraditório. Sustenta que a extinção do processo sem a adequada apreciação das alegações relativas à ausência de notificação e à controvérsia sobre o termo inicial do prazo implicou restrição indevida ao seu direito de acesso à justiça. Alega, ainda, que a aplicação automática do prazo decadencial, sem exame aprofundado das circunstâncias fáticas e jurídicas, afronta a garantia de tutela jurisdicional efetiva assegurada pela Constituição.
>>>

Resuma o texto jurídico acima.
"""
    }
]

In [ ]:
temperatura = 0.1
top_p = 0.9

resposta = chat_generate(
    prompt11,
    temperature = temperatura,
    top_p = top_p,
    max_tokens=2000
)

In [ ]:
printmd(resposta)

**SEÇÃO A - Segmentação Temática**

1. **Garantia Fundamental de Acesso à Jurisdição**
 * O artigo 5º, inciso XXXV, da Constituição Federal assegura o acesso à jurisdição para todos os cidadãos.
 * A cláusula pétrea garante a inafastabilidade da jurisdição, impedindo que atos administrativos ou legais obstem o controle judicial de ilegalidades ou abusos.

2. **Requerimento Administrativo e Decisão Administrativa**
 * O autor formulou requerimento administrativo para reconhecimento de direito.
 * A autoridade administrativa negou provimento ao pedido, fundamentando na ocorrência de prescrição.

3. **Controvérsia sobre Notificação e Prazo**
 * O autor sustenta que não foi regularmente notificado, comprometendo sua ciência inequívoca dos fatos e fundamentos.
 * A ausência de notificação válida inviabilizaria a análise do mérito do pleito.

4. **Decisão Judicial e Recurso**
 * O juízo de primeiro grau entendeu que não se tratava propriamente de prescrição, mas de decadência do direito.
 * A decisão judicial extinguiu o processo com resolução de mérito, aplicando o prazo quinquenal previsto no artigo 54 do Código de Defesa do Consumidor.

5. **Recurso e Alegações do Autor**
 * O autor afirma que a decisão judicial incorreu em violação a princípios constitucionais basilares.
 * Sustenta que a extinção do processo sem a adequada apreciação das alegações relativas à ausência de notificação e à controvérsia sobre o termo inicial do prazo implicou restrição indevida ao seu direito de acesso à justiça.

**SEÇÃO B - Classificação de Densidade**

1. **Garantia Fundamental de Acesso à Jurisdição** (Alta)
 * Núcleo argumentativo da garantia fundamental de acesso à jurisdição.

2. **Requerimento Administrativo e Decisão Administrativa** (Média)
 * Contexto relevante para a controvérsia sobre notificação e prazo.

3. **Controvérsia sobre Notificação e Prazo** (Alta)
 * Núcleo argumentativo da controvérsia sobre notificação e prazo.

4. **Decisão Judicial e Recurso** (Média)
 * Contexto relevante para a decisão judicial e recurso.

5. **Recurso e Alegações do Autor** (Alta)
 * Núcleo argumentativo das alegações do autor.

**SEÇÃO C - Iterações de Resumo**

**Resumo 1 (300 palavras)**

O autor formulou requerimento administrativo para reconhecimento de direito, mas a autoridade administrativa negou provimento ao pedido, fundamentando na ocorrência de prescrição. O autor sustenta que não foi regularmente notificado, comprometendo sua ciência inequívoca dos fatos e fundamentos. A ausência de notificação válida inviabilizaria a análise do mérito do pleito. O juízo de primeiro grau entendeu que não se tratava propriamente de prescrição, mas de decadência do direito, aplicando o prazo quinquenal previsto no artigo 54 do Código de Defesa do Consumidor. A decisão judicial extinguiu o processo com resolução de mérito. O autor afirma que a decisão judicial incorreu em violação a princípios constitucionais basilares, notadamente o devido processo legal, a ampla defesa e o contraditório.

**Resumo 2 (250 palavras)**

O autor formulou requerimento administrativo para reconhecimento de direito, mas a autoridade administrativa negou provimento ao pedido, fundamentando na ocorrência de prescrição. O autor sustenta que não foi regularmente notificado, comprometendo sua ciência inequívoca dos fatos e fundamentos. O juízo de primeiro grau entendeu que não se tratava propriamente de prescrição, mas de decadência do direito, aplicando o prazo quinquenal previsto no artigo 54 do Código de Defesa do Consumidor. A decisão judicial extinguiu o processo com resolução de mérito. O autor afirma que a decisão judicial incorreu em violação a princípios constitucionais basilares.

**Resumo 3 (200 palavras)**

O autor formulou requerimento administrativo para reconhecimento de direito, mas a autoridade administrativa negou provimento ao pedido, fundamentando na ocorrência de prescrição. O autor sustenta que não foi regularmente notificado, comprometendo sua ciência inequívoca dos fatos e fundamentos. O juízo de primeiro grau entendeu que não se tratava propriamente de prescrição, mas de decadência do direito, aplicando o prazo quinquenal previsto no artigo 54 do Código de Defesa do Consumidor. A decisão judicial extinguiu o processo com resolução de mérito.

**Resumo 4 (150 palavras)**

O autor formulou requerimento administrativo para reconhecimento de direito, mas a autoridade administrativa negou provimento ao pedido, fundamentando na ocorrência de prescrição. O autor sustenta que não foi regularmente notificado, comprometendo sua ciência inequívoca dos fatos e fundamentos. O juízo de primeiro grau entendeu que não se tratava propriamente de prescrição, mas de decadência do direito, aplicando o prazo quinquenal previsto no artigo 54 do Código de Defesa do Consumidor.

**Resumo 5 (100 palavras)**

O autor formulou requerimento administrativo para reconhecimento de direito, mas a autoridade administrativa negou provimento ao pedido, fundamentando na ocorrência de prescrição. O autor sustenta que não foi regularmente notificado, comprometendo sua ciência inequívoca dos fatos e fundamentos. O juízo de primeiro grau entendeu que não se tratava propriamente de prescrição, mas de decadência do direito.